# Part 3: Full Schema Synchronization PRO → PRE

This notebook synchronizes Hive objects between PRO and PRE environments with full support for:
1. EXTERNAL tables: automatic LOCATION update (PRO → PRE storage)
2. Views: detection and creation with adapted DDL
3. Partitioned tables: correct handling of partition columns
4. Schema updates: multiple ADD COLUMNS in a single ALTER TABLE
5. Data type changes: compatibility-based strategies

IMPORTANT:
- This notebook MUST be executed in the PRE environment
- It consumes outputs from Part 1 (DDL registry) and Part 2 (comparison results)

1. Configuración

In [ ]:

from pyspark.sql.functions import col
from datetime import datetime
import json
import re

CONFIG = {
    # Metadata tables
    'comparison_table': 'metadata.migration_validation_results',
    'ddl_registry_table': 'metadata.ddl_registry_pro',
    'sync_report_table': 'metadata.sync_execution_log',

    # Execution control
    'dry_run': True,  # ⚠️ Set to False to execute real changes
    'current_environment': 'PRE',

    # Storage configuration
    # These values are used to adapt EXTERNAL table locations
    'storage_pro': 'storage_pro_account',
    'storage_pre': 'storage_pre_account',

    # Table exclusion patterns (regex)
    'exclude_table_patterns': [
        r'.*bkp.*', r'.*backup.*', r'.*temp.*', r'.*tmp.*',
        r'.*clone.*', r'.*test.*', r'.*_old.*', r'.*dev.*', r'.*copy.*'
    ],

    # Database exclusion patterns (regex)
    'exclude_database_patterns': [
        r'.*backup.*', r'.*bkp.*', r'.*temp.*', r'.*tmp.*',
        r'.*test.*', r'.*_old.*', r'.*dev.*'
    ]
}


print("=" * 80)
print("PRO → PRE SCHEMA SYNCHRONIZATION ENGINE")
print("=" * 80)
print(f"Mode: {'DRY RUN (simulation only)' if CONFIG['dry_run'] else '⚠️ REAL EXECUTION'}")
print(f"Comparison table: {CONFIG['comparison_table']}")
print(f"DDL registry table: {CONFIG['ddl_registry_table']}")
print(f"Storage mapping: {CONFIG['storage_pro']} → {CONFIG['storage_pre']}")
print("=" * 80)

if CONFIG['dry_run']:
    print("\n⚠️ DRY RUN MODE ENABLED")
    print("No SQL statements will be executed")
    print("Set dry_run=False to apply changes\n")

2.  DDL analysis utilities

In [ ]:

def detect_table_type(ddl):
    """
    Detects the object type based on the DDL.

    Determines whether the object is:
    - VIEW
    - EXTERNAL TABLE
    - PARTITIONED TABLE
    - MANAGED TABLE

    Args:
        ddl (str): Full CREATE statement

    Returns:
        dict: Object classification flags
    """
    if not ddl:
        return {
            'type': 'UNKNOWN',
            'is_view': False,
            'is_external': False,
            'is_partitioned': False
        }

    ddl_upper = ddl.upper().strip()

    # Detect view
    is_view = ddl_upper.startswith('CREATE VIEW') or ddl_upper.startswith('CREATE OR REPLACE VIEW')

    # Detect external table
    is_external = 'EXTERNAL TABLE' in ddl_upper

    # Detect partitioned table
    is_partitioned = 'PARTITIONED BY' in ddl_upper

    if is_view:
        obj_type = 'VIEW'
    elif is_external:
        obj_type = 'EXTERNAL_TABLE'
    elif is_partitioned:
        obj_type = 'PARTITIONED_TABLE'
    else:
        obj_type = 'MANAGED_TABLE'

    return {
        'type': obj_type,
        'is_view': is_view,
        'is_external': is_external,
        'is_partitioned': is_partitioned
    }

2. Partition & LOCATION Extraction

In [ ]:
def extract_partition_columns(ddl):
    """
    Extracts partition column names from a CREATE TABLE DDL.

    Args:
        ddl (str): Full CREATE TABLE statement

    Returns:
        list[str]: Partition column names
    """
    if not ddl:
        return []

    match = re.search(r'PARTITIONED BY\s*\(([^)]+)\)', ddl, re.IGNORECASE | re.DOTALL)

    if not match:
        return []

    partition_clause = match.group(1)
    partition_columns = []

    for part in partition_clause.split(','):
        tokens = part.strip().split()
        if tokens:
            partition_columns.append(tokens[0].strip('`'))

    return partition_columns


def extract_location(ddl):
    """
    Extracts the LOCATION path from a DDL if present.

    Args:
        ddl (str): Full CREATE statement

    Returns:
        str | None: Storage path
    """
    if not ddl:
        return None

    match = re.search(r"LOCATION\s+['\"]([^'\"]+)['\"]", ddl, re.IGNORECASE)
    return match.group(1) if match else None

4. DDL Adaptation for PRE

In [ ]:
def adapt_ddl_for_pre(ddl, storage_pro, storage_pre):
    """
    Adapts a PRO DDL to PRE by replacing storage locations.

    This is mainly used for EXTERNAL tables to guarantee
    environment isolation.

    Args:
        ddl (str): Original PRO DDL
        storage_pro (str): PRO storage identifier
        storage_pre (str): PRE storage identifier

    Returns:
        str: Adapted DDL
    """
    if not ddl:
        return None

    adapted = ddl

    if storage_pro in adapted:
        adapted = adapted.replace(storage_pro, storage_pre)

    return adapted


5. Statement Generation (Create / Alter)

In [ ]:
def generate_create_statement(database, table_name, ddl, storage_pro, storage_pre):
    """
    Generates a CREATE statement for PRE based on PRO DDL.

    Handles:
    - Views
    - External tables (with adapted LOCATION)
    - Partitioned tables

    Returns:
        dict: Statement, object info, notes, errors
    """
    if not ddl:
        return {'statement': None, 'error': 'Empty or missing DDL'}

    table_info = detect_table_type(ddl)
    adapted_ddl = adapt_ddl_for_pre(ddl, storage_pro, storage_pre)

    notes = []

    if table_info['is_view']:
        notes.append('👁️ View detected')

    if table_info['is_external']:
        notes.append('📁 External table detected')
        location = extract_location(adapted_ddl)
        if location:
            notes.append(f'Location: {location}')

    if table_info['is_partitioned']:
        partitions = extract_partition_columns(adapted_ddl)
        if partitions:
            notes.append(f'Partitions: {", ".join(partitions)}')

    return {
        'statement': adapted_ddl,
        'table_info': table_info,
        'notes': notes,
        'error': None
    }


6. Schema Change Statements

In [ ]:
def generate_multi_add_columns_statement(database, table_name, columns_to_add):
    """
    Generates a single ALTER TABLE ADD COLUMNS statement.

    Args:
        columns_to_add (list): [{'name': str, 'type': str}]
    """
    if not columns_to_add:
        return None

    full_table = f"{database}.{table_name}"
    column_defs = [f"`{c['name']}` {c['type']}" for c in columns_to_add]

    return f"ALTER TABLE {full_table} ADD COLUMNS ({', '.join(column_defs)})"


7. Type Change Strategy

In [ ]:
def generate_change_type_statement(database, table_name, column_name, old_type, new_type):
    """
    Generates a column type change strategy.

    - Compatible changes use ALTER COLUMN
    - Incompatible changes generate a manual migration plan

    Returns:
        dict: SQL statement and strategy metadata
    """
    full_table = f"{database}.{table_name}"

    old_clean = old_type.lower().strip()
    new_clean = new_type.lower().strip()

    compatible_changes = [
        ('int', 'bigint'),
        ('float', 'double'),
        ('string', 'varchar'),
        ('varchar', 'string'),
    ]

    is_compatible = any(
        old_clean.startswith(o) and new_clean.startswith(n)
        for o, n in compatible_changes
    ) or 'decimal' in new_clean

    if is_compatible:
        stmt = f"ALTER TABLE {full_table} CHANGE COLUMN `{column_name}` `{column_name}` {new_type}"
        strategy = "ALTER_COLUMN"
    else:
        stmt = f"""
-- ⚠️ COMPLEX TYPE CHANGE: {column_name} ({old_type} → {new_type})
-- Manual migration required.
"""
        strategy = "MANUAL_MIGRATION"

    return {
        'statement': stmt,
        'strategy': strategy,
        'requires_manual': not is_compatible
    }
